# Laboratoire 02 : Ingestion Bronze Multi-Sources & Profilage Qualité Initial
**Cours :** Business Intelligence et Analyse des Mégadonnées (`01 31 0367 00`)  
**Public cible :** Étudiants ingénieurs en 4e année B.Sc. (Génie Industriel et Management de la Production)  
**Durée :** 3 Heures (180 Minutes)  
**Prérequis :** Laboratoire 01 (`.venv` et échauffement Python validés)

---

## 🎯 1. Contexte & Objectif Architectural
Dans les systèmes industriels, la donnée brute n'arrive jamais dans un format unique. Chaque cas d'étude du cours respecte une parité stricte de **3 formats hétérogènes** :
- **SmartPress Automotive (Cas de référence) :** `sensor_telemetry.csv` (SCADA), `maintenance_logs.json` (GMAO), `machines_and_shifts.xlsx` (ERP : `Machines`, `Shifts`, `Product_Variants`)
- **DataCo Hydraulics :** `order_items_trans.csv`, `shipment_events.json`, `master_catalog.xlsx`
- **Nexis Workforce :** `shift_production_performance.csv`, `workforce_events.json`, `workforce_roster_and_skills.xlsx`

Dans ce laboratoire, vous incarnez l'**Ingénieur Data** construisant la **Couche Bronze** :
1. Résoudre dynamiquement les chemins sur les dépôts étudiant et enseignant.
2. Ingérer les sources **CSV**, **JSON** et **Excel multi-onglets** avec `pandas`.
3. Injecter les métadonnées d'audit d'entreprise (`_ingestion_timestamp_utc`, `_source_file`).
4. Déverser les tables immuables au format **Parquet Snappy** dans `sandbox_outputs/<cas>/bronze_staged/` (`index=False`).
5. Profiler schémas, doublons, casse/espaces, valeurs nulles et extrêmes **sans aucune suppression** (le nettoyage s'effectuera aux **Labos 03–04**).

---

## ⚙️ 2. Configuration & Découverte Dynamique des Chemins

### Étape 1 : Contrôle des Bibliothèques & Localisation des Données Brutes

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case1_smartpress_automotive"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    return REF_CASE_DATA / stage_name
import json
from datetime import datetime, timezone
import pandas as pd

BRONZE_STAGED = SANDBOX_DATA / "bronze_staged"
BRONZE_STAGED.mkdir(parents=True, exist_ok=True)

print("Racine du dépôt :", REPO_ROOT)
print("Dossier Bronze  :", BRONZE_RAW, "| Existe :", BRONZE_RAW.exists())
print("Zone de Staging :", BRONZE_STAGED)

assert BRONZE_RAW.exists(), f"Dossier de données brutes introuvable : {BRONZE_RAW}"
for p in sorted(BRONZE_RAW.iterdir()):
    print(f"  - {p.name:<32} ({p.stat().st_size / 1024:,.1f} Ko)")


## 📥 3. Partie 1 : Ingestion du Flux CSV de Télémesure & Traçabilité d'Audit
Chargez `sensor_telemetry.csv`, ajoutez `_ingestion_timestamp_utc` et `_source_file`, puis vérifiez `.shape` et `.dtypes`.

In [ ]:
csv_name = "sensor_telemetry.csv"
now_utc = datetime.now(timezone.utc).isoformat()

# À FAIRE : Charger le CSV via pd.read_csv(BRONZE_RAW / csv_name)
df_telemetry = None  # À REMPLACER

# À FAIRE : Ajouter _ingestion_timestamp_utc = now_utc et _source_file = csv_name

if df_telemetry is not None:
    print("Dimensions Télémesure :", df_telemetry.shape)
    print(df_telemetry.head(3).to_string())


## 📋 4. Partie 2 : Ingestion des Journaux GMAO Semi-Structurés JSON
Analysez `maintenance_logs.json` avec `json.load()` et convertissez-le en `df_maintenance` avec colonnes d'audit.

In [ ]:
json_name = "maintenance_logs.json"

# À FAIRE : Lire BRONZE_RAW / json_name, convertir en DataFrame et ajouter les colonnes d'audit
df_maintenance = None  # À REMPLACER

if df_maintenance is not None:
    print("Dimensions Maintenance :", df_maintenance.shape)
    print(df_maintenance.head(3).to_string())


## 📊 5. Partie 3 : Ingestion du Classeur Excel Multi-Onglets (`Machines`, `Shifts`, `Product_Variants`)
Listez les onglets de `machines_and_shifts.xlsx` et chargez chaque entité métier séparément.

In [ ]:
excel_name = "machines_and_shifts.xlsx"
excel_path = BRONZE_RAW / excel_name

# À FAIRE : Inspecter pd.ExcelFile(excel_path).sheet_names et lire les 3 onglets
df_machines = None          # À REMPLACER
df_shifts = None            # À REMPLACER
df_product_variants = None  # À REMPLACER

# À FAIRE : Attacher _ingestion_timestamp_utc et _source_file sur chaque table


## 💾 6. Partie 4 : Exportation au Format Colonnaire Parquet (`index=False`)
Écrivez les 5 tables dans `BRONZE_STAGED/*.parquet` en excluant l'index entier Pandas.

In [ ]:
# À FAIRE : Exporter les 5 DataFrames vers BRONZE_STAGED avec index=False et compression="snappy"
outputs = {
    "bronze_telemetry.parquet": df_telemetry,
    "bronze_maintenance.parquet": df_maintenance,
    "bronze_machines.parquet": df_machines,
    "bronze_shifts.parquet": df_shifts,
    "bronze_product_variants.parquet": df_product_variants,
}

# ÉCRIRE VOTRE BOUCLE D'EXPORTATION ICI


## 🔍 7. Profilage Exploratoire Initial (Diagnostic sans Modification)
Relisez `bronze_telemetry.parquet` et `bronze_maintenance.parquet` pour inventorier les anomalies avant leur traitement aux **Labos 03–04**.

In [ ]:
# À FAIRE : Inspecter les machine_id uniques, les bornes de torque_Nm, les NaN, les doublons et downtime_type
